# 03_baseline_logistic_optuna
Logistic Regression + Optunaによるハイパーパラメータ最適化

In [1]:
%load_ext autoreload
%autoreload 2
import datetime, os, sys
from pathlib import Path
import numpy as np
import pandas as pd
PROJECT_ROOT = Path(
    "/Users/hayashikaito/Library/CloudStorage/GoogleDrive-1122hkaito@gmail.com/マイドライブ/jaggle_2026"
)
sys.path.append(str(PROJECT_ROOT))
from common.logistic.logistic_model import run_logistic
from common.logistic.logistic_model_optuna import run_logistic_optuna
from common.utils.logger import get_logger
from common.utils.metrics import calculate_logloss
from common.utils.seed import seed_everything
SEED = 42
seed_everything(seed=SEED)
TARGET_COL = "10年定着ラベル"
ID_COL = "社員ID"

In [2]:
SCRIPT_NAME = "03_baseline_logistic_optuna"
TODAY = datetime.datetime.now().strftime("%Y%m%d")
LOG_DIR = PROJECT_ROOT / "logs"
logger = get_logger(SCRIPT_NAME, log_dir=str(LOG_DIR))
logger.info(f"=== [{SCRIPT_NAME}] 実験開始 ===")

OUTPUT_DIR = PROJECT_ROOT / "data" / "output" / TODAY
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SUBMISSION_PATH = OUTPUT_DIR / f"{TODAY}_{SCRIPT_NAME}.csv"

SAVED_MODELS_DIR = PROJECT_ROOT / "saved_models" / TODAY / SCRIPT_NAME
SAVED_MODELS_DIR.mkdir(parents=True, exist_ok=True)

[2026-08-05 11:31:26] [INFO] === [03_baseline_logistic_optuna] 実験開始 ===


In [3]:
INPUT_DIR = PROJECT_ROOT / "data" / "input"
train_persona = pd.read_csv(INPUT_DIR / "employee_persona_train.csv")
test_persona = pd.read_csv(INPUT_DIR / "employee_persona_test.csv")
train_monthly = pd.read_csv(INPUT_DIR / "employee_monthly_train.csv")
test_monthly = pd.read_csv(INPUT_DIR / "employee_monthly_test.csv")

In [4]:
def transform_monthly_to_wide_all(monthly_df):
    val_cols = [col for col in monthly_df.columns if col not in ["社員ID", "経過月数"]]
    monthly_wide = monthly_df.pivot(index="社員ID", columns="経過月数", values=val_cols)
    monthly_wide.columns = [f"{col}_m{month}" for col, month in monthly_wide.columns]
    return monthly_wide.reset_index()
train_monthly_wide = transform_monthly_to_wide_all(train_monthly)
test_monthly_wide = transform_monthly_to_wide_all(test_monthly)
train_df = pd.merge(train_persona, train_monthly_wide, on="社員ID", how="left")
test_df = pd.merge(test_persona, test_monthly_wide, on="社員ID", how="left")

In [5]:
def build_features(train, test, target_col, id_col):
    train_proc, test_proc = train.copy(), test.copy()
    non_num_cols = train_proc.select_dtypes(include=["object"]).columns.tolist()
    if id_col in non_num_cols:
        non_num_cols.remove(id_col)
    train_proc = train_proc.drop(columns=non_num_cols, errors="ignore")
    test_proc = test_proc.drop(columns=non_num_cols, errors="ignore")
    X_train = train_proc.drop(columns=[target_col, id_col], errors="ignore")
    y_train = train_proc[target_col]
    X_test = test_proc.drop(columns=[id_col], errors="ignore")
    return {"X_train": X_train, "y_train": y_train, "X_test": X_test}, test_proc[id_col]
input_data, test_ids = build_features(train_df, test_df, TARGET_COL, ID_COL)

In [6]:
logistic_opt_params = {
    "n_splits": 5,
    "seed": SEED,
    "save_dir": str(SAVED_MODELS_DIR),
    "n_trials": 20,
}
logger.info("--- Optuna Logistic Regression チューニング開始 ---")
opt_logistic_res, best_logistic_params = run_logistic_optuna(data=input_data, params=logistic_opt_params)
logger.info(f"Logistic Regression Optuna Best Score: {opt_logistic_res['best_score']:.4f}")

[2026-08-05 11:31:26] [INFO] --- Optuna Logistic Regression チューニング開始 ---


/Users/hayashikaito/Library/CloudStorage/GoogleDrive-1122hkaito@gmail.com/マイドライブ/jaggle_2026/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/hayashikaito/Library/CloudStorage/GoogleDrive-1122hkaito@gmail.com/マイドライブ/jaggle_2026/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/hayashikaito/Library/CloudStorage/GoogleDrive-1122hkaito@gmail.com/マイドライブ/jaggle_2026/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/hayashikaito/Library/CloudStorage/GoogleDrive-1122hkaito@gmail.com/マイドライブ/jaggle_2026/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/hayashikaito/Library/CloudStorage/GoogleDrive-1122hkaito@gmail.com/マイドライブ/jaggle_2026/.venv/li

[2026-08-05 11:31:27] [INFO] Logistic Regression Optuna Best Score: 0.6733


/Users/hayashikaito/Library/CloudStorage/GoogleDrive-1122hkaito@gmail.com/マイドライブ/jaggle_2026/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/hayashikaito/Library/CloudStorage/GoogleDrive-1122hkaito@gmail.com/マイドライブ/jaggle_2026/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/hayashikaito/Library/CloudStorage/GoogleDrive-1122hkaito@gmail.com/マイドライブ/jaggle_2026/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/hayashikaito/Library/CloudStorage/GoogleDrive-1122hkaito@gmail.com/マイドライブ/jaggle_2026/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/hayashikaito/Library/CloudStorage/GoogleDrive-1122hkaito@gmail.com/マイドライブ/jaggle_2026/.venv/li

In [7]:
logger.info("--- Logistic Regression 本学習実行 ---")
final_params = {k: v for k, v in best_logistic_params.items() if k != "n_trials"}
logistic_res, _ = run_logistic(data=input_data, params=final_params)
logistic_cv = calculate_logloss(input_data["y_train"], logistic_res["oof_preds"])
logger.info(f"Logistic Regression CV Score: {logistic_cv:.4f}")

[2026-08-05 11:31:27] [INFO] --- Logistic Regression 本学習実行 ---
[2026-08-05 11:31:27] [INFO] Logistic Regression CV Score: 0.6733


/Users/hayashikaito/Library/CloudStorage/GoogleDrive-1122hkaito@gmail.com/マイドライブ/jaggle_2026/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/hayashikaito/Library/CloudStorage/GoogleDrive-1122hkaito@gmail.com/マイドライブ/jaggle_2026/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/hayashikaito/Library/CloudStorage/GoogleDrive-1122hkaito@gmail.com/マイドライブ/jaggle_2026/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/hayashikaito/Library/CloudStorage/GoogleDrive-1122hkaito@gmail.com/マイドライブ/jaggle_2026/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in m

In [8]:
sub = pd.DataFrame({ID_COL: test_ids, TARGET_COL: logistic_res["test_preds"]})
sub.to_csv(SUBMISSION_PATH, index=False)
logger.info(f"提出ファイル保存: {SUBMISSION_PATH}")
logger.info("=== 実験完了 ===")
print(f"\nOptuna Best: {opt_logistic_res['best_score']:.4f}")
print(f"Final CV: {logistic_cv:.4f}")

[2026-08-05 11:31:27] [INFO] 提出ファイル保存: /Users/hayashikaito/Library/CloudStorage/GoogleDrive-1122hkaito@gmail.com/マイドライブ/jaggle_2026/data/output/20260805/20260805_03_baseline_logistic_optuna.csv
[2026-08-05 11:31:27] [INFO] === 実験完了 ===

Optuna Best: 0.6733
Final CV: 0.6733
